# Unsupervised eccentricity-aware clustering

This notebook extends the unsupervised clustering workflow from **two features** to **three**:
- **CD206 brightness**
- **Cell area**
- **Cell eccentricity**

The clustering still runs **per donor**, but for cross-donor comparability it now uses a **strict reference-donor prototype mode**. The selected reference donor is clustered first in a shared standardized 3D feature space, and the other donor is then assigned against those established reference centroids instead of learning a new independent centroid geometry.

Because 3D scatter plots are a poor presentation format, the results are shown as **donor-specific pair plots** with points colored by the aligned cluster ID.


In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "macrophage_analysis").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import ipywidgets as widgets
from IPython.display import display

import macrophage_analysis as ma
from macrophage_analysis.analysis.clustering import (
    build_aligned_cluster_composition_table,
    ordered_present_values,
    run_split_kmeans,
)
from macrophage_analysis.analysis.morphology_tables import build_morphology_table
from macrophage_analysis.notebook import (
    build_clustering_feature_labels,
    build_marker_dropdown,
)
from macrophage_analysis.plotting.clustering import (
    plot_aligned_cluster_composition,
    plot_cluster_feature_pairs,
)

cluster_donors = list(ma.DEFAULT_DONORS)
cluster_conditions = list(ma.DEFAULT_CONDITIONS)
cluster_antibody_dropdown = build_marker_dropdown()
cluster_feature_columns = ("intensity", "area", "eccentricity")
cluster_feature_label_extras = {
    "area": "Area",
    "eccentricity": "Eccentricity",
}
cluster_centroid_columns = (
    "cd206_intensity_centroid",
    "area_centroid",
    "eccentricity_centroid",
)
cluster_non_reference_mode = "independent"
cluster_max_alignment_distance = 2.0
cluster_n_tiles = (8, 8)
pairplot_max_points_per_donor = 3000

display(cluster_antibody_dropdown)
cluster_donors, cluster_conditions, cluster_antibody_dropdown.value, cluster_feature_columns


Run this once. The extraction is still focused on **CD206** because the three clustering features are all derived from the same measurement path: intensity, area, and eccentricity.


In [ ]:
cluster_antibody = str(cluster_antibody_dropdown.value)
cluster_feature_labels = build_clustering_feature_labels(
    cluster_antibody,
    extra_labels=cluster_feature_label_extras,
)

cluster_results = ma.extract_single_cell_fluorescence(
    donors=cluster_donors,
    conditions=cluster_conditions,
    antibody_order=[cluster_antibody],
    n_tiles=cluster_n_tiles,
)


Build the per-cell feature table that the clustering will use. This is a quick QC checkpoint so you can confirm the donors and treatments have enough cells and the eccentricity signal is actually present.


In [ ]:
cluster_cells = build_morphology_table(
    cluster_results,
    antibody=cluster_antibody,
)

display(
    cluster_cells.groupby(["donor_label", "condition_label"], observed=True)
    .agg(
        cell_count=("cell_index", "count"),
        median_intensity=("intensity", "median"),
        median_area=("area", "median"),
        median_eccentricity=("eccentricity", "median"),
    )
    .reset_index()
)


## Clustering parameters

Choose `K` and the **reference donor**. The reference donor is clustered first, and the other donor is then assigned against those fixed reference centroids in the same standardized 3-feature space. This keeps the cluster identities directly comparable across donors.


In [ ]:
final_cluster_k = 3
final_cluster_seed = 666
final_cluster_iterations = 15

reference_donor_options = ordered_present_values(cluster_cells["donor_label"])
reference_donor_dropdown = widgets.Dropdown(
    options=reference_donor_options,
    value=reference_donor_options[0],
    description="Reference donor",
)
display(reference_donor_dropdown)


Run the actual **3-feature K-means clustering** here. The centroid table below is useful for checking whether the clusters make biological sense in terms of brightness, size, and roundness.


In [ ]:
reference_donor_label = str(reference_donor_dropdown.value)

(
    donor_feature_data,
    donor_runs,
    clustered_cells,
    cluster_centroid_table,
) = run_split_kmeans(
    cluster_cells,
    k=final_cluster_k,
    random_seed=final_cluster_seed,
    max_iterations=final_cluster_iterations,
    reference_donor_label=reference_donor_label,
    feature_columns=cluster_feature_columns,
    centroid_column_names=cluster_centroid_columns,
    non_reference_mode=cluster_non_reference_mode,
    max_alignment_distance=cluster_max_alignment_distance,
)

display(
    cluster_centroid_table[[
        "reference_donor_label",
        "donor_label",
        "cluster",
        "aligned_cluster",
        "alignment_distance",
        "cd206_intensity_centroid",
        "area_centroid",
        "eccentricity_centroid",
    ]].round({
        "alignment_distance": 3,
        "cd206_intensity_centroid": 2,
        "area_centroid": 1,
        "eccentricity_centroid": 3,
    })
)


## Pair plots

This is the key visual output. The clustering math was done in **3D**, but the result is shown as donor-specific **pair plots** for every 2D projection:
- brightness vs area
- brightness vs eccentricity
- area vs eccentricity

The dots are colored by the **aligned cluster ID**, so you can inspect how cleanly the clusters separate across all three feature pairs.


In [ ]:
plot_cluster_feature_pairs(
    clustered_cells,
    feature_columns=cluster_feature_columns,
    feature_labels=cluster_feature_labels,
    cluster_column="aligned_cluster",
    max_points_per_donor=pairplot_max_points_per_donor,
)


## Treatment composition

After the unsupervised clustering is done, this shows which treatments are enriched in each aligned hidden cluster.


In [ ]:
cluster_composition = build_aligned_cluster_composition_table(clustered_cells)

display(cluster_composition)
plot_aligned_cluster_composition(
    cluster_composition,
    reference_donor_label=reference_donor_label,
    cluster_count=final_cluster_k,
)
